# Buoy detector v2: retrain and compare YOLO11 vs YOLO26 vs RF-DETR (transformer) + TensorRT

**Runtime:** *Runtime → Change runtime type → **A100*** (or L4). Then *Runtime → Run all*.
Private repo: add the Colab secret `GH_TOKEN` (key icon), as before.

| Step | What | Approx. time on A100 (estimate) |
|---|---|---|
| 1 | Setup + rebuild the cleaned dataset (frames cut from the repo videos) | 5 min |
| 2 | Train YOLO11s, 1280 px | 10–15 min |
| 3 | Train YOLO26s, 1280 px (NMS-free) | 10–15 min |
| 4 | Train RF-DETR Small, 896 px (DINOv2 transformer, Apache-2.0) | 25–40 min |
| 5 | Evaluate all models + the old T4 model + Grounding DINO on the same held-out videos | 5 min |
| 6 | TensorRT FP16 export + speed | 5 min |
| 7 | Full demo with the best model (course logic v2) | 10–15 min |
| 8 | Save a results zip to Google Drive | 2 min |

Every long step shows a **tqdm progress bar**; nothing is hidden. Set `RUN_*` flags below to skip steps.

In [ ]:
RUN_YOLO11, RUN_YOLO26, RUN_RFDETR = True, True, True
RUN_TENSORRT, RUN_DEMO = True, True
EPOCHS_YOLO, EPOCHS_RFDETR = 40, 40
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing; WORKERS = min(8, multiprocessing.cpu_count()); print('CPU cores:', multiprocessing.cpu_count())

In [ ]:
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/lytvynenkomaksim/CV_test.git" if tok else "https://github.com/lytvynenkomaksim/CV_test.git"
if not os.path.exists('/content/CV_test'):
    !git clone --depth 1 -b main {url} /content/CV_test
%cd /content/CV_test
!git log --oneline | head -3

## 1. Setup

In [ ]:
!apt-get -qq install -y libegl1 libgles2 > /dev/null
!pip install -q ultralytics transformers accelerate rtmlib onnxruntime-gpu lap mediapipe "rfdetr[train,loggers]" tqdm
!bash tools/download_models.sh
!cp weights/buoy_yolo11s.pt models/buoy_yolo11s_t4.pt   # the first (raw-label) model, as baseline
!yolo settings weights_dir=/content/CV_test/models > /dev/null
os.environ['HF_HOME'] = '/content/CV_test/models/hf'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# cleaned dataset v2: labels are in git, images are cut from the videos (progress bar)
!python tools/extract_label_frames.py --dataset data/buoy_dataset_v2
!cat data/buoy_dataset_v2/cleaning_stats.json

## 2–4. Training

In [ ]:
if RUN_YOLO11:
    !python tools/train_buoy_yolo.py --data data/buoy_dataset_v2/data.yaml --model yolo11s.pt --name buoy_v2_yolo11s \
        --epochs {EPOCHS_YOLO} --patience 10 --imgsz 1280 --batch 32 --workers {WORKERS} --cache ram
    !cp models/buoy_yolo11s.pt models/buoy_v2_yolo11s.pt

In [ ]:
if RUN_YOLO26:
    !python tools/train_buoy_yolo.py --data data/buoy_dataset_v2/data.yaml --model yolo26s.pt --name buoy_v2_yolo26s \
        --epochs {EPOCHS_YOLO} --patience 10 --imgsz 1280 --batch 32 --workers {WORKERS} --cache ram
    !cp models/buoy_yolo26s.pt models/buoy_v2_yolo26s.pt

In [ ]:
if RUN_RFDETR:
    !python tools/train_rfdetr.py --dataset data/buoy_dataset_v2 --variant RFDETRSmall --resolution 896 \
        --epochs {EPOCHS_RFDETR} --batch 8 --grad-accum 2 --workers {WORKERS}

## 5. Evaluate everything on the same held-out frames
`AP50`, precision/recall at the best-F1 threshold, `R_centre` (hit = centre within one buoy size), and ms per image on this GPU.

In [ ]:
models = ["yolo:models/buoy_yolo11s_t4.pt:1280"]
if os.path.exists('models/buoy_v2_yolo11s.pt'): models.append("yolo:models/buoy_v2_yolo11s.pt:1280")
if os.path.exists('models/buoy_v2_yolo26s.pt'): models.append("yolo:models/buoy_v2_yolo26s.pt:1280")
if os.path.exists('models/buoy_rfdetr.pth'): models.append("rfdetr:models/buoy_rfdetr.pth:RFDETRSmall:896")
models.append("gdino:-")
args = " ".join(f"--model {m}" for m in models)
!python tools/eval_detectors.py --dataset data/buoy_dataset_v2 {args} --out results/eval_v2

## 6. TensorRT FP16 (NVIDIA): same YOLO weights, compiled engine

In [ ]:
if RUN_TENSORRT:
    from ultralytics import YOLO
    for w in ['models/buoy_v2_yolo11s.pt', 'models/buoy_v2_yolo26s.pt']:
        if os.path.exists(w):
            YOLO(w).export(format='engine', half=True, imgsz=1280, device=0)
    eng = [f"--model yolo:{w}:1280" for w in ['models/buoy_v2_yolo11s.engine', 'models/buoy_v2_yolo26s.engine'] if os.path.exists(w)]
    !python tools/eval_detectors.py --dataset data/buoy_dataset_v2 {" ".join(eng)} --out results/eval_v2_tensorrt

## 7. Full demo with the best detector (course logic v2: gates, 6 slots, hidden-buoy projection)

In [ ]:
import json
ev = json.load(open('results/eval_v2/eval.json'))
cands = {k: v for k, v in ev.items() if 'gdino' not in k and 't4' not in k}
best = max(cands, key=lambda k: cands[k]['AP50']); print('best:', best, cands[best])
if RUN_DEMO:
    if 'rfdetr' in best or 'checkpoint' in best:
        buoy = 'rfdetr'
    else:
        !cp models/{'buoy_v2_yolo26s.pt' if 'yolo26' in best else 'buoy_v2_yolo11s.pt'} models/buoy_yolo11s.pt
        buoy = 'yolo-buoy'
    !python tools/run_all.py --buoy {buoy} --pose yolo-s --out runs/demo_v2

## 8. Save results to Google Drive
Then share `MyDrive/cv_test_results_v2/` (Anyone with the link → Viewer) and paste the link in the chat.

In [ ]:
!mkdir -p results_v2 && cp -r results/eval_v2* results_v2/ 2>/dev/null; cp models/buoy_v2_* models/buoy_rfdetr.* results_v2/ 2>/dev/null
!for d in runs/train/buoy_v2_yolo11s runs/train/buoy_v2_yolo26s; do [ -d $d ] && mkdir -p results_v2/$(basename $d) && cp $d/results.csv $d/*.png results_v2/$(basename $d)/; done
![ -d runs/train/buoy_rfdetr ] && mkdir -p results_v2/buoy_rfdetr && cp runs/train/buoy_rfdetr/metrics.csv runs/train/buoy_rfdetr/training_config.json results_v2/buoy_rfdetr/
!cp -r runs/demo_v2 results_v2/ 2>/dev/null
!cd results_v2 && zip -qr ../cv_test_results_v2.zip . && cd .. && ls -lh cv_test_results_v2.zip
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cv_test_results_v2 && cp cv_test_results_v2.zip /content/drive/MyDrive/cv_test_results_v2/
print('saved to MyDrive/cv_test_results_v2/cv_test_results_v2.zip')